
# Детекция объектов на видео: пример с "L'arrivée d'un train" (Братья Люмьер)

В этом ноутбуке мы реализуем базовую детекцию объектов на примере известного ролика "Прибытие поезда" братьев Люмьер.  
Для детекции используем метод обнаружения движущихся объектов с помощью фоновой субстракции.


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display



## Загрузка и отображение видео

Загрузим видеофайл. Вы можете заменить путь на локальный файл `train_arrival.mp4`.


In [ ]:

video_path = 'train_arrival.mp4'  # Убедитесь, что файл лежит в рабочей директории
display(Video(video_path, embed=True))



## Детекция объектов с помощью фоновой субстракции

Используем простой метод — вычитание фона (`cv2.createBackgroundSubtractorMOG2`), чтобы выделить движущиеся объекты.


In [ ]:

cap = cv2.VideoCapture(video_path)
fgbg = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=50)
detected_frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    fgmask = fgbg.apply(frame)
    
    # Наложим маску на исходный кадр
    contours, _ = cv2.findContours(fgmask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        if cv2.contourArea(cnt) > 500:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
    detected_frames.append(frame)

cap.release()
print(f"Обработано кадров: {len(detected_frames)}")



## Отображение кадров с выделенными объектами


In [ ]:

for i in range(3):
    rgb = cv2.cvtColor(detected_frames[i], cv2.COLOR_BGR2RGB)
    plt.imshow(rgb)
    plt.title(f'Кадр {i+1} с детекцией')
    plt.axis('off')
    plt.show()



## Сохранение видео с детекцией


In [ ]:

out_path = 'train_detection_output.avi'
height, width, _ = detected_frames[0].shape
out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'XVID'), 20.0, (width, height))

for frame in detected_frames:
    out.write(frame)
out.release()

print("Видео сохранено как", out_path)
